In [ ]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"] = os.path.join("/home/workspace/z_model_pretrained","gitee-ai")


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import cv2
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler,StableDiffusionPipeline
from diffusers.utils import load_image,make_image_grid
import numpy as np
import torch
from transformers import AutoImageProcessor, UperNetForSemanticSegmentation
from peft import LoraConfig, get_peft_model
import random
import ptp_utils
import abc
from torchvision.utils import save_image

LOW_RESOURCE = False

In [4]:

class AttentionControl(abc.ABC):
    
    def step_callback(self, x_t):
        return x_t
    
    def between_steps(self):
        return
    
    @property
    def num_uncond_att_layers(self):
        return self.num_att_layers if LOW_RESOURCE else 0
    
    @abc.abstractmethod
    def forward (self, attn, is_cross: bool, place_in_unet: str):
        raise NotImplementedError

    def __call__(self, attn, is_cross: bool, place_in_unet: str):
        # print(1)
        if self.cur_att_layer >= self.num_uncond_att_layers:
            # print(2)
            if LOW_RESOURCE:
                attn = self.forward(attn, is_cross, place_in_unet)
            else:
                h = attn.shape[0]
                attn[h // 2:] = self.forward(attn[h // 2:], is_cross, place_in_unet)
        self.cur_att_layer += 1
        if self.cur_att_layer == self.num_att_layers + self.num_uncond_att_layers:
            self.cur_att_layer = 0
            self.cur_step += 1
            self.between_steps()
        return attn
    
    def reset(self):
        self.cur_step = 0
        self.cur_att_layer = 0

    def __init__(self):
        self.cur_step = 0
        self.num_att_layers = -1
        self.cur_att_layer = 0

# do nothing
class EmptyControl(AttentionControl):
    
    def forward (self, attn, is_cross: bool, place_in_unet: str):
        return attn
    

class AttentionStore(AttentionControl):
    @staticmethod
    def get_empty_store():
        return {"down_cross": [], "up_cross": []}

    def forward(self, attn, is_cross: bool, place_in_unet: str):
        if is_cross and attn.shape[1] == 16 ** 2:  # 16x16 token
            key = f"{place_in_unet}_cross"
            heads = 8
            tmp = attn.clone().detach().view(-1, heads, *attn.shape[-2:]).mean(1)
            self.step_store[key].append(tmp)
        return attn

    def between_steps(self):
        if not self.step_store["down_cross"] and not self.step_store["up_cross"]:
            self.step_store = self.get_empty_store()
            return

        all_maps = self.step_store["down_cross"] + self.step_store["up_cross"]
        layer_avg = torch.stack(all_maps, dim=0).mean(0)
        cond_map = layer_avg[0]


        H = W = 16
        for tok_id in range(cond_map.shape[-1]):
            img = cond_map[:, tok_id].view(H, W)
            img_min = img.min(dim=0, keepdim=True)[0].min(dim=1, keepdim=True)[0]
            img_max = img.max(dim=0, keepdim=True)[0].max(dim=1, keepdim=True)[0]
            img = (img - img_min) / (img_max - img_min)

            save_image(img.unsqueeze(0).unsqueeze(0),
                       os.path.join(self.save_dir, f"tok_{tok_id:02d}_step_{self.cur_step-1:02d}.png"))

        self.step_store = self.get_empty_store()

    def get_current_attention(self):
        return self.step_store

    def reset(self):
        super().reset()
        self.step_store = self.get_empty_store()

    def __init__(self,save_dir):
        super().__init__()
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.step_store = self.get_empty_store()

In [5]:
#Available modes Pose, Depth, HED, Canny, Seg

# pose,depth,canny,hed,seg,normal
controlnet_libs = {
    'pose': "lllyasviel/sd-controlnet-openpose",
    'scribble': "lllyasviel/sd-controlnet-scribble",
    'canny': "lllyasviel/sd-controlnet-canny",
    'hed': "lllyasviel/sd-controlnet-hed",
    'depth': "lllyasviel/sd-controlnet-depth",
    'seg' : "lllyasviel/sd-controlnet-seg",
    'normal': "lllyasviel/sd-controlnet-normal",
    'mlsd': "lllyasviel/sd-controlnet-mlsd",
}

def load_pipeline(mode_1, mode_2,device="cuda:0"):

    controlnet_1 = ControlNetModel.from_pretrained(controlnet_libs[mode_1],torch_dtype=torch.float32,local_files_only=False)
    controlnet_2 = ControlNetModel.from_pretrained(controlnet_libs[mode_2],torch_dtype=torch.float32,local_files_only=False)
    
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet_2,torch_dtype=torch.float32)

    # pipe.enable_freeu(s1=0.9, s2=0.2, b1=1.5, b2=1.6)
    
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

    pipe.controlnet1=controlnet_1.to(device)
    pipe.controlnet2= controlnet_2.to(device)

    pipe=pipe.to(device)

    return pipe



In [ ]:
mode_1 = 'pose'
mode_2 = 'depth'
pipe= load_pipeline(mode_1, mode_2)

In [7]:
seed= 42
g_cpu = torch.Generator().manual_seed(seed)
latent = torch.randn((1, 4,512 // 8, 512 // 8),generator=g_cpu,)

In [ ]:
control1_path = "./conditions/6.png"
control2_path = "./conditions/5.png"
prompt = "A man in grey walking on the street , harmoniously , best quality"

prompts=[prompt]
negative_prompts=[" monochrome, bad anatomy, lowres,  worst quality, low quality"]

control1=load_image(control1_path)
control2=load_image(control2_path)


save_dir="attn_maps"
controller = AttentionStore(save_dir)

# Index of the foreground token in the prompt
token_index = 2

thres = 1/256

with torch.amp.autocast("cuda"):
    image, x_t = ptp_utils.text2image_ldm_stable(pipe, 
                                                    prompts, 
                                                    negative_prompts, 
                                                    latent=latent, 
                                                    num_inference_steps=50, 
                                                    guidance_scale=7.5, 
                                                    generator=g_cpu, 
                                                    control1=control1,
                                                    control2=control2,
                                                    low_resource=LOW_RESOURCE,
                                                    thres=thres,
                                                    controller=controller,
                                                    token_index=token_index,
                                                    dir=save_dir)

ptp_utils.view_images(images=image,results_dir="./results")